# 04b · Token-level encoder (E2): where was the text rewritten? · GPU

E1 labels a whole window, so a fusion text with one rewritten sentence teaches it that unchanged human text can be "fusion", and averaging windows dilutes a short rewrite. In validation, X4 calls 44% of fusion texts with ≤ 5% rewritten words human.

E2 tags every token as *original*, *rewritten* or *ai*. The labels come from train fusion texts whose human source is known (`splits/fusion_sources.csv`, the TF-IDF neighbour pairs from `02_split`): a word is *rewritten* when none of its word 3-grams occurs in the source. Words of human texts are *original*, words of ai texts *ai*; fusion texts without a known source are left out of training. Fusion training windows contain a rewritten token with probability `focus`. Prediction covers the whole text with overlapping windows and summarises the token probabilities per text (means, max of a 16-token moving average, share above 0.5, position of the rewritten peak); the stacker in `07` uses these.

Ways to run:
- local GPU machine: `uv run python -m hw3.tokens`, then open this notebook from the report cells;
- Colab / Kaggle: run this notebook; the first cell clones the repo (push `splits/fusion_sources.csv` first). On Colab set the Kaggle token as a secret;
- `SMOKE = True`: tiny CPU run that only checks the pipeline; outputs go to `artifacts/smoke/`.

After a cloud run, download the zip from the last cell and extract it into `artifacts/` of the local repo.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


REPO_URL = "https://github.com/kerfmanner/ai-text-detection-hw3-ucu-ai.git"
CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if CLOUD:
    if not Path("ai-text-detection-hw3-ucu-ai").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("ai-text-detection-hw3-ucu-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.24.0"], check=True)
    sys.path.insert(0, str(Path("src").resolve()))
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CLOUD

In [ ]:
SMOKE = False

import pandas as pd
import torch

from hw3 import config, tokens
from hw3.data import download, splits
from hw3.runs import compare, export_outputs, report, val_report
from hw3.sentences import fusion_sources

config.set_smoke(SMOKE)
download()
cfg = config.load_config("tokens")
torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", cfg

In [ ]:
info = tokens.run(cfg)
info["train_tokens"], info["token_f1_val"]

Standalone scores use a simple rule (ai = mean ai probability, fusion = (1 − ai) × smoothed max rewritten probability); the stacker in `07` learns from all summaries.

In [ ]:
rep = report("E2")
rep["figure"].savefig(config.ROOT / "images" / "confusion_E2.png", dpi=150, bbox_inches="tight")
rep["per_class"]

Validation fusion texts with a known source, by share of rewritten words: how often each model calls them human.

In [ ]:
sources, text = fusion_sources(), splits()[1].set_index("id").text
r = val_report("E2")
f = r[(r.label == "fusion") & r.id.isin(sources.index)]
f = f.assign(rewritten=[tokens.word_tags(text[i].split(), "fusion", sources[i]).mean() for i in f.id])
f.groupby(pd.cut(f.rewritten, [0, 0.05, 0.1, 0.2, 1.0]), observed=True).agg(n=("id", "size"), called_human=("pred", lambda p: (p == "human").mean()), mean_rewritten_smax=("rewritten_smax", "mean")).round(3)

In [ ]:
compare(["B0", "E1", "E2"])

In [ ]:
export_outputs("E2_outputs", ["preds/E2_*", "runs/E2*", "logs/E2*"])